<a href="https://colab.research.google.com/github/Rixdorfer-Kindl/protein-design-pipline-colab/blob/main/%E6%9A%B4%E9%9C%B2%E6%96%91%E5%9D%97%E5%88%86%E6%9E%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title 🧭 6ARU Domain III 暴露斑块分析 { display-mode: "form" }
!pip install -q biopython
import numpy as np, pandas as pd
from scipy.spatial.distance import cdist
from Bio.PDB import PDBParser, PDBList, ShrakeRupley
from Bio.Data.IUPACData import protein_letters_3to1

pdb_id = "6aru" #@param {type:"string"}
chain_id = "A" #@param {type:"string"}
dom_start = 310 #@param {type:"integer"}
dom_end = 481 #@param {type:"integer"}
rsa_cutoff = 0.30 #@param {type:"number"}
patch_radius = 10.0 #@param {type:"number"}

MAXASA = {"ALA":129,"ARG":274,"ASN":195,"ASP":193,"CYS":167,"GLN":225,"GLU":223,"GLY":104,
          "HIS":224,"ILE":197,"LEU":201,"LYS":236,"MET":224,"PHE":240,"PRO":159,"SER":155,
          "THR":172,"TRP":285,"TYR":263,"VAL":174}
p = PDBList().retrieve_pdb_file(pdb_id, pdir="/content", file_format="pdb")
model = PDBParser(QUIET=True).get_structure("s", p)[0]
print("链:", {c.id: len(list(c)) for c in model})

for c in list(model):
    if c.id != chain_id: model.detach_child(c.id)
for r in list(model[chain_id]):
    if r.get_resname() in ("HOH", "WAT"): model[chain_id].detach_child(r.id)
ShrakeRupley().compute(model, level="R")

res = {}
for r in model[chain_id]:
    if r.id[0] == " " and r.get_resname() in MAXASA:
        ca = r["CA"].coord if "CA" in r else None
        res[r.id[1]] = dict(aa=protein_letters_3to1[r.get_resname().capitalize()],
                            name=r.get_resname(), rsa=r.sasa/MAXASA[r.get_resname()],
                            sasa=r.sasa, ca=ca,
                            xyz=np.array([a.coord for a in r if a.element != "H"]))

seq = "".join(v["aa"] for v in res.values()); nums = list(res)
sequon = [nums[i] for i in range(len(nums)-2)
          if seq[i]=="N" and seq[i+1]!="P" and seq[i+2] in "ST"]
glyc_xyz = [res[n]["xyz"] for n in sequon]
def near_glycan(n):
    return any(cdist(res[n]["xyz"], g).min() < 6.0 for g in glyc_xyz) if glyc_xyz else False

dom = [n for n in nums if dom_start <= n <= dom_end]
exposed = [n for n in dom if res[n]["rsa"] > rsa_cutoff and not near_glycan(n)]
print(f"Domain 内残基 {len(dom)}，暴露且避开糖基化 {len(exposed)}，sequon Asn: {sequon}")

rows = []
for c in exposed:
    nb = [n for n in exposed if np.linalg.norm(res[n]["ca"]-res[c]["ca"]) < patch_radius]
    aas = [res[n]["aa"] for n in nb]
    rows.append({"center": c, "aa": res[c]["aa"], "n_res": len(nb),
                 "area_A2": round(sum(res[n]["sasa"] for n in nb)),
                 "疏水/芳香": sum(a in "FWYLIMV" for a in aas),
                 "负电 D/E": sum(a in "DE" for a in aas),
                 "正电 K/R": sum(a in "KR" for a in aas),
                 "members": nb})
df = pd.DataFrame(rows)
df["score"] = df["area_A2"]/100 + 2*df["疏水/芳香"]   # 粗略打分，权重是我随手定的
df = df.sort_values("score", ascending=False)
pd.set_option("display.width", 250, "display.max_colwidth", 80)
display(df.drop(columns="members").head(15))
df.to_pickle("/content/patches_6aru.pkl")

链: {'A': 611, 'B': 210, 'C': 213, 'D': 8, 'E': 2}
Domain 内残基 172，暴露且避开糖基化 50，sequon Asn: [104, 151, 172, 328, 337, 389, 420, 504, 544, 579, 599]


,center,aa,n_res,area_A2,疏水/芳香,负电 D/E,正电 K/R,score
38,464,T,8,756,1,0,3,9.56
43,472,E,9,734,1,1,1,9.34
31,454,K,9,891,0,1,3,8.91
10,356,S,5,677,1,0,1,8.77
45,476,K,8,670,1,1,1,8.70
46,477,A,8,670,1,1,1,8.70
42,471,G,7,649,1,1,1,8.49
47,478,T,7,636,1,1,1,8.36
3,320,E,6,621,1,2,1,8.21
7,342,S,5,419,2,0,1,8.19


In [2]:
#@title 🎯 西妥昔单抗表位 → 6ARU hotspot 候选 { display-mode: "form" }
!pip install -q biopython
import numpy as np, pandas as pd, re
from scipy.spatial.distance import cdist
from Bio.PDB import PDBParser, PDBList, ShrakeRupley
from Bio.Align import PairwiseAligner
from Bio.Data.IUPACData import protein_letters_3to1

#@markdown ### 参考复合物 (1YY9) 与目标 (6ARU)
cx_pdb = "1yy9" #@param {type:"string"}
egfr_chain_cx = "A" #@param {type:"string"}
tgt_pdb = "6aru" #@param {type:"string"}
tgt_chain = "A" #@param {type:"string"}
dom_start = 310 #@param {type:"integer"}
dom_end = 481 #@param {type:"integer"}
contact_cutoff = 5.0 #@param {type:"number"}
rsa_cutoff = 0.30 #@param {type:"number"}

MAXASA = {"ALA":129,"ARG":274,"ASN":195,"ASP":193,"CYS":167,"GLN":225,"GLU":223,"GLY":104,
          "HIS":224,"ILE":197,"LEU":201,"LYS":236,"MET":224,"PHE":240,"PRO":159,"SER":155,
          "THR":172,"TRP":285,"TYR":263,"VAL":174}
parser = PDBParser(QUIET=True)

def load(pid):
    p = PDBList().retrieve_pdb_file(pid, pdir="/content", file_format="pdb")
    return parser.get_structure(pid, p)[0]

def reslist(chain):
    out = []
    for r in chain:
        if r.id[0] == " " and r.get_resname().capitalize() in protein_letters_3to1:
            out.append((r.id[1], protein_letters_3to1[r.get_resname().capitalize()], r))
    return out

def rsa_map(model, chain_id):
    """只保留指定链(含糖,去水)，算残基相对SASA"""
    for c in list(model):
        if c.id != chain_id: model.detach_child(c.id)
    for r in list(model[chain_id]):
        if r.get_resname() in ("HOH", "WAT"): model[chain_id].detach_child(r.id)
    ShrakeRupley().compute(model, level="R")
    return {r.id[1]: r.sasa / MAXASA[r.get_resname()]
            for r in model[chain_id] if r.id[0] == " " and r.get_resname() in MAXASA}

# ---------- 1YY9：界面 ----------
cx = load(cx_pdb)
print("1YY9 各链长度:", {c.id: len(reslist(c)) for c in cx})
egfr = reslist(cx[egfr_chain_cx])
ab_atoms = [a for c in cx if c.id != egfr_chain_cx and len(reslist(c)) >= 50
            for r in c if r.id[0] == " " for a in r if a.element != "H"]
assert ab_atoms, "没找到抗体链，检查 egfr_chain_cx"
ab_xyz = np.array([a.coord for a in ab_atoms])

iface = {}
for n, aa, r in egfr:
    xyz = np.array([a.coord for a in r if a.element != "H"])
    D = cdist(xyz, ab_xyz)
    k = int((D < contact_cutoff).sum())
    if k: iface[n] = (aa, k, round(float(D.min()), 2))

# ---------- 暴露度 (去掉抗体后的完整 EGFR 链) ----------
rsa_cx = rsa_map(load(cx_pdb), egfr_chain_cx)

# ---------- 6ARU ----------
tg_model = load(tgt_pdb)
tg = reslist(tg_model[tgt_chain])
tg_aa = {n: a for n, a, _ in tg}
rsa_tg = rsa_map(load(tgt_pdb), tgt_chain)

# ---------- 比对映射 ----------
al = PairwiseAligner(); al.mode = "local"
al.match_score, al.mismatch_score = 2, -1
al.open_gap_score, al.extend_gap_score = -5, -0.5
a_seq = "".join(a for _, a, _ in egfr); b_seq = "".join(a for _, a, _ in tg)
aln = al.align(a_seq, b_seq)[0]
m = {}
for (r0, r1), (s0, s1) in zip(*aln.aligned):
    for k in range(r1 - r0):
        m[egfr[r0+k][0]] = (tg[s0+k][0], egfr[r0+k][1] == tg[s0+k][1])

# ---------- 糖基化 sequon (6ARU 序列) ----------
sequon_N = [tg[i][0] for i in range(len(tg)-2)
            if b_seq[i] == "N" and b_seq[i+1] != "P" and b_seq[i+2] in "ST"]
sequon_pos = set(sequon_N) | {n+2 for n in sequon_N}
N_xyz = {n: np.array([a.coord for r in [x for x in tg if x[0]==n][0][2:] for a in r]) for n in sequon_N}
all_res = {n: np.array([a.coord for a in r if a.element != "H"]) for n, _, r in tg}
near_glycan = {n for n, xyz in all_res.items()
               if any(cdist(xyz, N_xyz[s]).min() < 6.0 for s in sequon_N)}

# ---------- 汇总 ----------
rows = []
for n, (aa, k, dmin) in iface.items():
    if n not in m: continue
    tn, same = m[n]
    rows.append({"1YY9_num": n, "aa": aa, "6ARU_num": tn, "aa一致": same,
                 "ab_contacts": k, "min_dist": dmin,
                 "RSA_1YY9(无抗体)": round(rsa_cx.get(n, np.nan), 2),
                 "RSA_6ARU": round(rsa_tg.get(tn, np.nan), 2),
                 "疏水/芳香": aa in "FWYLIMV",
                 "sequon": tn in sequon_pos, "近糖基化": tn in near_glycan,
                 "在domain内": dom_start <= tn <= dom_end})
df = pd.DataFrame(rows).sort_values("ab_contacts", ascending=False)
pd.set_option("display.width", 250, "display.max_columns", 20)
print(f"\n6ARU 链{tgt_chain} sequon 位置(Asn): {sequon_N}")
display(df)

cand = df[(df["RSA_1YY9(无抗体)"] > rsa_cutoff) & df["aa一致"] & df["在domain内"]
          & ~df["sequon"] & ~df["近糖基化"]]
print(f"\n候选 hotspot (暴露 + 在domain内 + 避开糖基化): {len(cand)} 个")
display(cand)
top = cand.sort_values(["疏水/芳香", "ab_contacts"], ascending=False).head(6)
print("\nppi.hotspot_res=[" + ",".join(f"{tgt_chain}{n}" for n in sorted(top['6ARU_num'])) + "]")
df.to_csv("/content/cetuximab_epitope_6ARU.csv", index=False)

1YY9 各链长度: {'A': 613, 'C': 211, 'D': 220, 'B': 0, 'E': 0, 'F': 0, 'G': 0}
Structure exists: '/content/pdb1yy9.ent' 
Structure exists: '/content/pdb6aru.ent' 
Structure exists: '/content/pdb6aru.ent' 

6ARU 链A sequon 位置(Asn): [104, 151, 172, 328, 337, 389, 420, 504, 544, 579, 599]


,1YY9_num,aa,6ARU_num,aa一致,ab_contacts,min_dist,RSA_1YY9(无抗体),RSA_6ARU,疏水/芳香,sequon,近糖基化,在domain内
19,468,S,468,True,73,2.60,0.56,0.59,False,False,False,True
18,467,I,467,True,59,3.33,0.37,0.38,True,False,False,True
15,443,K,443,True,52,3.04,0.59,0.55,False,False,True,True
6,409,H,409,True,47,3.36,0.64,0.65,False,False,False,True
20,469,N,469,True,32,2.90,0.20,0.19,False,False,False,True
0,349,P,349,True,25,3.32,0.31,0.32,False,False,False,True
16,465,K,465,True,25,2.87,0.47,0.47,False,False,False,True
24,473,N,473,True,22,3.11,0.69,0.60,False,False,False,True
4,384,Q,384,True,22,2.75,0.33,0.34,False,False,False,True
10,417,V,417,True,22,3.55,0.29,0.26,True,False,False,True



候选 hotspot (暴露 + 在domain内 + 避开糖基化): 11 个


,1YY9_num,aa,6ARU_num,aa一致,ab_contacts,min_dist,RSA_1YY9(无抗体),RSA_6ARU,疏水/芳香,sequon,近糖基化,在domain内
19,468,S,468,True,73,2.60,0.56,0.59,False,False,False,True
18,467,I,467,True,59,3.33,0.37,0.38,True,False,False,True
6,409,H,409,True,47,3.36,0.64,0.65,False,False,False,True
0,349,P,349,True,25,3.32,0.31,0.32,False,False,False,True
16,465,K,465,True,25,2.87,0.47,0.47,False,False,False,True
24,473,N,473,True,22,3.11,0.69,0.60,False,False,False,True
4,384,Q,384,True,22,2.75,0.33,0.34,False,False,False,True
2,353,R,353,True,12,3.17,0.59,0.60,False,False,False,True
22,471,G,471,True,12,3.40,0.33,0.32,False,False,False,True
7,411,Q,411,True,6,3.49,0.41,0.40,False,False,False,True



ppi.hotspot_res=[A349,A409,A465,A467,A468,A473]


In [3]:
import numpy as np, itertools
from Bio.PDB import PDBParser
m = PDBParser(QUIET=True).get_structure("s", "/content/pdb6aru.ent")[0]["A"]
cand = [349, 353, 384, 409, 411, 465, 467, 468, 471, 472, 473]
ca = {n: m[n]["CA"].coord for n in cand}
import pandas as pd
D = pd.DataFrame([[np.linalg.norm(ca[a]-ca[b]) for b in cand] for a in cand],
                 index=cand, columns=cand).round(1)
display(D)

,349,353,384,409,411,465,467,468,471,472,473
349,0.000000,6.200000,6.400000,18.400000,20.799999,21.799999,20.500000,19.600000,24.900000,27.100000,29.799999
353,6.200000,0.000000,12.500000,23.900000,26.799999,27.900000,26.100000,24.700001,29.299999,31.700001,34.000000
384,6.400000,12.500000,0.000000,14.500000,15.800000,15.600000,14.900000,14.800000,20.400000,22.100000,25.200001
409,18.400000,23.900000,14.500000,0.000000,5.100000,15.200000,18.799999,21.000000,28.200001,28.900000,32.400002
411,20.799999,26.799999,15.800000,5.100000,0.000000,11.900000,16.700001,19.600000,26.100000,26.200001,29.799999
465,21.799999,27.900000,15.600000,15.200000,11.900000,0.000000,6.200000,9.900000,15.000000,14.700000,18.200001
467,20.500000,26.100000,14.900000,18.799999,16.700001,6.200000,0.000000,3.800000,9.800000,10.800000,13.800000
468,19.600000,24.700001,14.800000,21.000000,19.600000,9.900000,3.800000,0.000000,8.000000,10.300000,12.700000
471,24.900000,29.299999,20.400000,28.200001,26.100000,15.000000,9.800000,8.000000,0.000000,3.800000,5.100000
472,27.100000,31.700001,22.100000,28.900000,26.200001,14.700000,10.800000,10.300000,3.800000,0.000000,3.800000


In [4]:
import numpy as np
from Bio.PDB import PDBParser
ch = PDBParser(QUIET=True).get_structure("s", "/content/pdb6aru.ent")[0]["A"]  # 文件名按实际改
hs = [467, 468, 471, 472]
hs_xyz = np.array([a.coord for n in hs for a in ch[n]])
keep = sorted(r.id[1] for r in ch if 310 <= r.id[1] <= 481 and r.id[0] == " "
              and np.linalg.norm(np.array([a.coord for a in r])[:, None] - hs_xyz[None], axis=2).min() < 20)
print(keep)


[345, 346, 347, 348, 349, 350, 352, 353, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 396, 399, 402, 404, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481]
